In [ ]:
from lsst.summit.utils import ConsDbClient

In [ ]:
import numpy as np
from astropy.table import Table, join
from astropy.time import Time

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
%matplotlib widget

import pandas as pd

from lsst.meas.algorithms.installGaussianPsf import FwhmPerSigma

from tqdm.notebook import tqdm

import os
import pandas as pd
from matplotlib.cm import get_cmap

In [ ]:
os.environ["no_proxy"] += ",.consdb"

In [ ]:
url="http://consdb-pq.consdb:8080/consdb"

In [ ]:
consdb=ConsDbClient(url)

In [ ]:
# Query both consDB tables
exposure = consdb.query("SELECT * FROM cdb_lsstcam.exposure WHERE science_program in ('BLOCK-T614') AND day_obs > 20251020 AND observation_reason in ('aos_stability_test', 'infocus_aos_stability_test', 'extra_aos_stability_test', 'intra_aos_stability_test')")

visits_ql = consdb.query("SELECT * FROM cdb_lsstcam.visit1_quicklook")

# Join using astropy's join function on 'visit_id'
exposure_join = exposure.rename_column("exposure_id", "visit_id")
merged_visits = join(exposure, visits_ql, keys="visit_id", join_type="inner")  

In [ ]:
len(exposure)

In [ ]:
len(merged_visits)

In [ ]:
exposure.columns

In [ ]:
merged_visits.columns

In [ ]:
import pandas as pd
import numpy as np

# ============================================================
# 1. Convert input table to pandas DataFrame
# ============================================================
try:
    df = merged_visits.to_pandas()
except Exception:
    df = merged_visits.copy()

# ============================================================
# 2. Basic time columns and sorting
# ============================================================
df['night'] = pd.to_datetime(df['day_obs_1'].astype(str), format='%Y%m%d')
df['obs_start_dt'] = pd.to_datetime(df['obs_start'], errors='coerce', utc=True)
df = df.sort_values(['night', 'obs_start_dt'])

# ============================================================
# 3. Assign test_index based on time gaps
# ============================================================
gap_seconds = 5 * 60  # 5 minutes threshold

def assign_test_index(group):
    group = group.copy()
    gaps = group['obs_start_dt'].diff().dt.total_seconds().fillna(0)
    group['test_index'] = (gaps > gap_seconds).cumsum()
    return group

df = df.groupby('night', group_keys=False).apply(assign_test_index)

df['test_id'] = df['night'].dt.strftime('%Y-%m-%d') + '_T' + df['test_index'].astype(str)

# ============================================================
# 4. Test start time
# ============================================================
test_start_times = (
    df.groupby(['night', 'test_index'])['obs_start_dt']
      .min()
      .rename('test_start_time')
)

# ============================================================
# 5. Aggregation: MEAN ONLY
# ============================================================
# --- NEW COLUMN: cleaned + floored physical rotator angle ---
df['physical_rotator_angle'] = pd.to_numeric(df['physical_rotator_angle'], errors='coerce')

df.loc[df['physical_rotator_angle'].abs() < 3, 'physical_rotator_angle'] = 0
df['physical_rotator_angle_floor'] = np.floor(df['physical_rotator_angle'])

# --- MEAN-ONLY aggregation block (updated) ---
agg_dict = {
    'altitude':     'mean',
    'azimuth':      'mean',
    'physical_rotator_angle_floor': 'mean',
    'wind_speed':   'mean',
    'wind_dir':     'mean',
}

test_stats = (
    df.groupby(['night', 'test_index'])
      .agg(agg_dict)
)

test_stats = test_stats.round(0)
test_stats.columns = [f"{col}_mean" for col in test_stats.columns]
test_stats = test_stats.reset_index()

# ============================================================
# 6. Add science_program per test
# (use most frequent program inside that test)
# ============================================================
science_program_per_test = (
    df.groupby(['night', 'test_index'])['science_program']
      .agg(lambda s: s.mode().iloc[0] if not s.mode().empty else None)
      .rename('science_program')
)

test_stats = test_stats.merge(
    science_program_per_test.reset_index(),
    on=['night', 'test_index'],
    how='left'
)

# ============================================================
# 7. Add test_start_time
# ============================================================
test_stats = test_stats.merge(
    test_start_times.reset_index(),
    on=['night', 'test_index'],
    how='left'
)

# Add readable columns
test_stats['night_str'] = test_stats['night'].dt.strftime('%Y-%m-%d')
test_stats['test_id'] = (
    test_stats['night_str'] + '_T' + test_stats['test_index'].astype(str)
)

# Reorder
cols_front = [
    'night_str', 'test_start_time',
    'science_program'
]
other_cols = [c for c in test_stats.columns if c not in cols_front and c not in ['night']]

test_stats = test_stats[cols_front + other_cols]

# ------------------------------------------------------------
# Rename output columns to human-friendly labels
# ------------------------------------------------------------
rename_map = {
    'night_str': 'Day Obs',
    'altitude_mean': 'Elevation',
    'azimuth_mean': 'Azimuth',
    'physical_rotator_angle_floor_mean': 'Rotator',
    'wind_speed_mean': 'Wind Spd',
    'wind_dir_mean': 'Wind Dir',
}

test_stats = test_stats.rename(columns=rename_map)

# ============================================================
# Count how many times each (Elevation, Azimuth, Rotator) combo occurred
# ============================================================
combo_counts = (
    test_stats
    .groupby(['Elevation', 'Azimuth', 'Rotator'])
    .size()
    .reset_index(name='Count')
)

# ============================================================
# 8. View clean, human-readable output
# ============================================================
test_stats

In [ ]:
combo_counts